In [101]:
import csv
from pathlib import Path
import inflect
import re

from pydantic import BaseModel
from pydantic_ai import Agent
from pydantic_ai.models.ollama import OllamaModel
from pydantic_ai.providers.ollama import OllamaProvider

In [102]:
model = OllamaModel(
    'llama3.1', provider=OllamaProvider(base_url='http://localhost:11434/v1')
)

stock_csv = Path('../resturaunt_files/stock.csv')

stock_agent = Agent(
    model,
    system_prompt=(
        'You are a stock assistant. For any stock or inventory question, always call lookup_stock '
        'or list_stock first, including non-food items like cash. Answer only from tool results.'
    ),
)


_p = inflect.engine()

def _normalize_name(name: str) -> str:
    """Normalize case/spacing and singularize so stock names are handled consistently."""
    n = re.sub(r"[\s_-]+", "-", (name or "").strip().lower())
    if not n:
        return n
    *head, last = n.split("-")          # only the head noun pluralizes
    return "-".join([*head, _p.singular_noun(last) or last])


def _load_stock_rows() -> list[dict]:
    with stock_csv.open(newline='', encoding='utf-8') as f:
        return list(csv.DictReader(f))


@stock_agent.tool_plain
def list_stock() -> list[dict]:
    """Return all stock rows as dictionaries (ingredient, qty, unit, reorder_threshold)."""
    return _load_stock_rows()


@stock_agent.tool_plain
def lookup_stock(item: str) -> dict:
    """Return stock details for any ingredient in stock.csv."""
    normalized_item = _normalize_name(item)

    for row in _load_stock_rows():
        ingredient_raw = row.get('ingredient') or ''
        ingredient = ingredient_raw.strip()
        if _normalize_name(ingredient) == normalized_item:
            qty_raw = (row.get('qty') or '0').strip()
            return {
                'ingredient': ingredient,
                'qty': float(qty_raw),
                'unit': (row.get('unit') or '').strip(),
                'reorder_threshold': float((row.get('reorder_threshold') or '0').strip()),
            }

    raise ValueError(f"Ingredient not found in stock.csv: {item}")

In [103]:
class StockCheck(BaseModel):
    stock: str
    qty: int

In [104]:
result = await stock_agent.run("How many burgers do we have?")

print(result.output)

We have 8 burgers in stock. Our reorder threshold for burgers is 5, so you may want to consider ordering more soon.


In [106]:
for question in [
    "How many burgers do we have?",
    "How much mustard is in stock?",
    "How many windows do we have?",
    "What is the stock entry for cash?",
]:
    result = await stock_agent.run(question)
    print(question, "->", result.output)

How many burgers do we have? -> According to the stock status, we have 8 burgers available. Our reorder threshold is set at 5, so we should restock soon.
How much mustard is in stock? -> We currently have 20 servings of mustard in stock. Our reorder threshold is 5 servings, so we will need to restock soon.


Traceback (most recent call last):
  File "c:\Users\devan\Agent_Resturaunt\.venv\Lib\site-packages\IPython\core\interactiveshell.py", line 3773, in run_code
    await eval(code_obj, self.user_global_ns, self.user_ns)
  File "C:\Users\devan\AppData\Local\Temp\ipykernel_13212\4019331360.py", line 7, in <module>
    result = await stock_agent.run(question)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\devan\Agent_Resturaunt\.venv\Lib\site-packages\pydantic_ai\agent\abstract.py", line 660, in run
    node = await agent_run.next(node)  # pyright: ignore[reportArgumentType]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\devan\Agent_Resturaunt\.venv\Lib\site-packages\pydantic_ai\run.py", line 470, in next
    return await self._run_node_with_hooks(node, self._stream_and_advance)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\devan\Agent_Resturaunt\.venv\Lib\site-packages\pydantic_ai\run.py", line 392, in _run_node_with_hooks